# Pressing analysis — Phase 1: opponent build-up phases

**Event unit:** a *build-up phase* = one team (A) has the ball with the ball in A's own or middle third.
Pressing is later described as what the other team (B) does during the phase, so passive phases are kept too.

A phase **starts** when A has (cleaned) possession, the frame is usable and the ball is in A's own/middle third.
It **ends** when possession changes, A reaches its attacking third (confirmed), a stoppage starts,
the camera cuts away, or the match ends.

Workflow:
1. Run all cells up to **Review sync** → `phases_auto.csv` and `phases_review.csv`.
2. Optionally export review clips (last section).
3. Fill `decision` / `edit_*` columns in `phases_review.csv`; add missed phases in `phases_added.csv`.
4. Run **Apply review** → `phases_final.parquet` (input for Phase 2).

Re-running with different thresholds keeps your review decisions (matched by team + frame overlap).

In [ ]:
from pathlib import Path
import shutil
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------- paths
CACHE = Path(r"C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)"
             r"\project\barca_atletico_first_half\new_cache")
PLAYER_PATH = CACHE / "player_frame_table_named.parquet"
BALL_PATH = CACHE / "ball_frame_table_named.parquet"
VIDEO_PATH = CACHE.parent / "barca_atletico_46m53s.mkv"   # <-- set to your video file (only used for clips)

OUT_DIR = CACHE / "pressing_phases"
AUTO_PATH = OUT_DIR / "phases_auto.csv"
REVIEW_PATH = OUT_DIR / "phases_review.csv"
ADDED_PATH = OUT_DIR / "phases_added.csv"
FINAL_PATH = OUT_DIR / "phases_final.parquet"
CLIP_DIR = OUT_DIR / "clips"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- pitch
PITCH_L, PITCH_W = 105.0, 68.0
OWN_THIRD_MAX = PITCH_L / 3          # 35 m
ATTACK_THIRD_MIN = 2 * PITCH_L / 3   # 70 m

# ---------------------------------------------------------------- frame validity
MIN_PEOPLE_VISIBLE = 16      # fewer boxes than this = close-up / replay -> frame unusable
INVALID_BRIDGE_SEC = 0.4     # bridge short unusable blips (single bad homography frames)

# ---------------------------------------------------------------- possession cleanup
POSS_NA_FILL_SEC = 1.0       # fill unknown possession runs <= this if same team on both sides
POSS_FLICKER_SEC = 0.5       # remove other-team flickers <= this sandwiched by the same team

# ---------------------------------------------------------------- ball position
BALL_FFILL_SEC = 1.0         # carry last known ball position at most this long
BALL_BOUNDS_MARGIN = 5.0     # positions further outside the pitch are treated as homography errors

# ---------------------------------------------------------------- zone hysteresis
ZONE_CONFIRM_SEC = 1.0       # entering / leaving the attacking third must last this long

# ---------------------------------------------------------------- phase assembly
MERGE_GAP_SEC = 1.0          # merge same-team phases split only by unusable / unknown frames
MIN_PHASE_SEC = 3.0

# ---------------------------------------------------------------- start / end labels (heuristic guesses, reviewed)
CONTEXT_SEC = 3.0                  # look-back / look-ahead window
RESTART_MIN_NOCARRIER_SEC = 3.0    # dead-ball gap before a restart
TOUCHLINE_MARGIN = 3.0             # ball this close to a touchline at a restart -> throw-in
CENTRE_RADIUS = 5.0                # ball this close to the centre spot at a restart -> kick-off
GK_RESTART_MAX_X_ATT = 20.0        # ball in own box area at a restart -> goalkeeper restart
BOX_HALF_WIDTH = 20.16
LEFT_IS_LOW_Y_ATT = True           # lane naming convention; verify once on a clip (see lanes note below)

# ---------------------------------------------------------------- quality flag
MIN_BALL_POS_FRAC = 0.6
MIN_DEF_VISIBLE_MEAN = 8.0

# ---------------------------------------------------------------- review
REVIEW_MATCH_IOU = 0.6
REVIEW_COLS = ["decision", "edit_team", "edit_start_frame", "edit_end_frame",
               "edit_start_type", "edit_end_reason", "note"]

## Load and check

In [ ]:
P_COLS = ["frame_idx", "track_id", "role", "team", "team_name", "name", "number",
          "x1", "y1", "x2", "y2", "pitch_x", "pitch_y", "attack_dir", "x_att"]
P = (pd.read_parquet(PLAYER_PATH, columns=P_COLS)
       .sort_values(["frame_idx", "track_id"]).reset_index(drop=True))
B = pd.read_parquet(BALL_PATH).sort_values("frame_idx").reset_index(drop=True)

last_frame = int(P.frame_idx.max())
n_extra = int((B.frame_idx > last_frame).sum())
if n_extra:
    print(f"dropping {n_extra} ball row(s) after the last player frame {last_frame}")
B = B[B.frame_idx <= last_frame].reset_index(drop=True)
assert (B.frame_idx.to_numpy() == np.arange(len(B))).all(), "ball table must have one row per frame from 0"

N = len(B)
FPS = float(round(1.0 / np.median(np.diff(B.t_sec.to_numpy(float)))))
def nfr(sec):
    return max(1, int(round(sec * FPS)))
print(f"frames: {N}, fps: {FPS}")

tn = P.dropna(subset=["team"]).drop_duplicates("team")
TEAM_NAME = {int(t): str(n) for t, n in zip(tn.team, tn.team_name)}
TEAMS = sorted(TEAM_NAME)
assert len(TEAMS) == 2, TEAM_NAME
OTHER = {TEAMS[0]: TEAMS[1], TEAMS[1]: TEAMS[0]}
NAME_TO_TEAM = {v: k for k, v in TEAM_NAME.items()}
print("teams:", TEAM_NAME)

ad = P.dropna(subset=["team", "attack_dir"])
ad_counts = ad.groupby("team")["attack_dir"].value_counts().unstack(fill_value=0)
print("attack_dir rows per team:\n", ad_counts)
ATTACK_DIR = {int(t): int(ad_counts.loc[t].idxmax()) for t in ad_counts.index}
assert ATTACK_DIR[TEAMS[0]] == -ATTACK_DIR[TEAMS[1]], "teams should attack opposite directions"

chk = ad.dropna(subset=["pitch_x", "x_att"])
chk = chk.sample(min(len(chk), 200_000), random_state=0)
recon = np.where(chk.attack_dir.astype(int).to_numpy() == 1,
                 chk.pitch_x.to_numpy(float), PITCH_L - chk.pitch_x.to_numpy(float))
err = np.abs(recon - chk.x_att.to_numpy(float))
print(f"x_att reconstruction error: median {np.median(err):.4f} m, p99 {np.percentile(err, 99):.4f} m")
assert np.percentile(err, 99) < 0.05, "x_att does not follow x if attack_dir==1 else L-x; check PITCH_L / convention"

ID_INFO = (P.drop_duplicates("track_id").set_index("track_id")[["name", "number", "role", "team"]]
             .to_dict("index"))

## Helpers

In [ ]:
def run_table(codes):
    """Runs of equal consecutive values -> DataFrame(start, end (inclusive), value, length)."""
    codes = np.asarray(codes)
    if len(codes) == 0:
        return pd.DataFrame(columns=["start", "end", "value", "length"])
    change = np.flatnonzero(codes[1:] != codes[:-1]) + 1
    starts = np.r_[0, change]
    ends = np.r_[change - 1, len(codes) - 1]
    return pd.DataFrame({"start": starts, "end": ends, "value": codes[starts],
                         "length": ends - starts + 1})


def replace_short_runs(codes, max_len, which):
    """Interior runs with length <= max_len for which which(value, prev, next) is True
    take the previous run's value (single pass)."""
    codes = np.asarray(codes).copy()
    rt = run_table(codes)
    st, en, val, ln = (rt[c].to_numpy() for c in ("start", "end", "value", "length"))
    for i in range(1, len(rt) - 1):
        if ln[i] <= max_len and which(val[i], val[i - 1], val[i + 1]):
            codes[st[i]:en[i] + 1] = val[i - 1]
    return codes


def as_bool(s):
    return s.astype("boolean").fillna(False).to_numpy(dtype=bool)


def as_code(s):
    """Nullable int column -> int array with -1 for missing."""
    return s.astype("Int64").fillna(-1).to_numpy(dtype=int)


def to_att(x, y, team):
    """Pitch coords -> coords in `team`'s attacking frame (0 = own goal line)."""
    d = ATTACK_DIR[int(team)]
    return (x, y) if d == 1 else (PITCH_L - x, PITCH_W - y)


def third_of(xa):
    if xa is None or np.isnan(xa):
        return None
    return "own" if xa < OWN_THIRD_MAX else ("middle" if xa < ATTACK_THIRD_MIN else "attacking")


LANES = ("left", "centre", "right") if LEFT_IS_LOW_Y_ATT else ("right", "centre", "left")
def lane_of(ya):
    if ya is None or np.isnan(ya):
        return None
    return LANES[0] if ya < PITCH_W / 3 else (LANES[1] if ya < 2 * PITCH_W / 3 else LANES[2])


def mmss(t):
    return f"{int(t // 60):02d}:{int(t % 60):02d}"


def interval_iou(a0, a1, b0, b1):
    inter = max(0, min(a1, b1) - max(a0, b0) + 1)
    union = (a1 - a0 + 1) + (b1 - b0 + 1) - inter
    return inter / union if union > 0 else 0.0

## Frame-level signals
Usable frames, cleaned possession, best-available ball position, and the possessing team's zone
(own/middle = build-up, attacking) with hysteresis.

In [ ]:
# ---- usable frames
stop = as_bool(B.is_stoppage)
hom_ok = as_bool(B.homography_valid)
n_vis = B.n_people_visible.to_numpy(int)
visual_ok_raw = (hom_ok & (n_vis >= MIN_PEOPLE_VISIBLE)).astype(int)
visual_ok = replace_short_runs(visual_ok_raw, nfr(INVALID_BRIDGE_SEC),
                               lambda v, p, n: v == 0 and p == 1 and n == 1) == 1
valid = visual_ok & ~stop

# ---- possession
poss_raw = as_code(B.possession_team)
poss = replace_short_runs(poss_raw, nfr(POSS_NA_FILL_SEC),
                          lambda v, p, n: v == -1 and p >= 0 and p == n)
poss = replace_short_runs(poss, nfr(POSS_FLICKER_SEC),
                          lambda v, p, n: v >= 0 and p >= 0 and p == n and v != p)

# ---- carrier
car_id = as_code(B.carrier_track_id)
car_team = as_code(B.carrier_team)
nocar_len = np.zeros(N, dtype=int)   # length of the no-carrier run ending at each frame
for i in range(N):
    nocar_len[i] = (nocar_len[i - 1] + 1 if i else 1) if car_id[i] < 0 else 0

# ---- ball position: real ball -> carrier position -> short forward fill
bx = B.ball_pitch_x.to_numpy(float).copy()
by = B.ball_pitch_y.to_numpy(float).copy()
ball_src = B.ball_source.astype(str).to_numpy()
M = BALL_BOUNDS_MARGIN
with np.errstate(invalid="ignore"):
    ok = (np.isin(ball_src, ["detected", "smoothed"]) & hom_ok & ~as_bool(B.ball_speed_implausible)
          & (bx > -M) & (bx < PITCH_L + M) & (by > -M) & (by < PITCH_W + M))
bx[~ok] = np.nan
by[~ok] = np.nan
pos_src = np.where(ok, "ball", "none").astype(object)

has_car = car_id >= 0
cf = pd.DataFrame({"frame_idx": np.flatnonzero(has_car), "track_id": car_id[has_car]})
cf = cf.merge(P[["frame_idx", "track_id", "pitch_x", "pitch_y"]], on=["frame_idx", "track_id"], how="left")
cx = np.full(N, np.nan)
cy = np.full(N, np.nan)
cx[cf.frame_idx.to_numpy()] = cf.pitch_x.to_numpy(float)
cy[cf.frame_idx.to_numpy()] = cf.pitch_y.to_numpy(float)
use = np.isnan(bx) & ~np.isnan(cx)
bx[use], by[use] = cx[use], cy[use]
pos_src[use] = "carrier"

filled = pd.DataFrame({"x": bx, "y": by}).ffill(limit=nfr(BALL_FFILL_SEC))
ff = np.isnan(bx) & filled.x.notna().to_numpy()
pos_src[ff] = "ffill"
bx, by = filled.x.to_numpy(), filled.y.to_numpy()

# ---- zone of the possessing team: 1 = build-up (own/middle third), 0 = attacking third, -1 = unknown
zone = np.full(N, -1, dtype=int)
for r in run_table(poss).itertuples():
    if r.value < 0:
        continue
    sl = slice(r.start, r.end + 1)
    xa, _ = to_att(bx[sl], by[sl], r.value)
    z = pd.Series(np.where(np.isnan(xa), np.nan, (xa < ATTACK_THIRD_MIN).astype(float)))
    z = z.ffill().bfill()
    if z.isna().all():
        continue
    z = z.to_numpy(int)
    z = replace_short_runs(z, nfr(ZONE_CONFIRM_SEC) - 1, lambda v, p, n: True)
    zone[sl] = z

key = np.where(valid & (poss >= 0) & (zone == 1), poss, -1)

pd.Series({
    "usable frames %": 100 * valid.mean(),
    "stoppage %": 100 * stop.mean(),
    "visual unusable % (after bridging)": 100 * (~visual_ok).mean(),
    "possession known % raw": 100 * (poss_raw >= 0).mean(),
    "possession known % cleaned": 100 * (poss >= 0).mean(),
    "ball pos from ball %": 100 * (pos_src == "ball").mean(),
    "ball pos from carrier %": 100 * (pos_src == "carrier").mean(),
    "ball pos ffill %": 100 * (pos_src == "ffill").mean(),
    "ball pos none %": 100 * (pos_src == "none").mean(),
    "build-up frames (any team) %": 100 * (key >= 0).mean(),
}).round(2)

## Phase segmentation

In [ ]:
def gap_is_soft(team, gs, ge):
    """A gap between two same-team candidate phases only caused by unusable frames / unknown possession."""
    if ge < gs:
        return True
    sl = slice(gs, ge + 1)
    if stop[sl].any():
        return False
    if (poss[sl] == OTHER[team]).any():
        return False
    if ((poss[sl] == team) & (zone[sl] == 0) & valid[sl]).any():
        return False
    return True


cand = run_table(key)
cand = cand[cand.value >= 0]
segments = []
for r in cand.itertuples():
    t = int(r.value)
    if segments and segments[-1]["team"] == t:
        gs, ge = segments[-1]["end"] + 1, int(r.start) - 1
        if (ge - gs + 1) <= nfr(MERGE_GAP_SEC) and gap_is_soft(t, gs, ge):
            segments[-1]["end"] = int(r.end)
            segments[-1]["n_merged"] += 1
            continue
    segments.append({"team": t, "start": int(r.start), "end": int(r.end), "n_merged": 0})

segments = [s for s in segments if (s["end"] - s["start"] + 1) >= nfr(MIN_PHASE_SEC)]
print(f"candidate runs: {len(cand)}, phases after merge + min duration: {len(segments)}")

## Start / end labels and phase descriptors
Start types: `kickoff`, `gk_restart` (goal kick or keeper restart in own box), `throw_in`, `other_restart`,
`regain`, `recycle` (came back from own attacking third), `after_cut`, `unknown`.
End reasons: `possession_lost`, `reached_attacking_third`, `stoppage`, `camera_cut`, `possession_unknown`, `match_end`.
These are heuristic guesses meant to be corrected in review.

In [ ]:
def first_known(arr, s, e):
    idx = np.flatnonzero(~np.isnan(arr[s:e + 1]))
    return s + idx[0] if len(idx) else None


def first_own_carrier(team, s, e):
    sl = slice(s, e + 1)
    idx = np.flatnonzero((car_id[sl] >= 0) & (car_team[sl] == team))
    return int(car_id[s + idx[0]]) if len(idx) else None


def classify_start(team, s):
    if s <= nfr(2.0):
        return "kickoff"
    ctx = slice(max(0, s - nfr(CONTEXT_SEC)), s)
    gap_sec = nocar_len[s - 1] / FPS
    if stop[ctx].any() or gap_sec >= RESTART_MIN_NOCARRIER_SEC:
        f = first_known(bx, s, min(N - 1, s + nfr(1.0)))
        x, y = (bx[f], by[f]) if f is not None else (np.nan, np.nan)
        xa, _ = to_att(x, y, team)
        fc = first_own_carrier(team, s, min(N - 1, s + nfr(3.0)))
        fc_role = ID_INFO.get(fc, {}).get("role") if fc is not None else None
        if np.hypot(x - PITCH_L / 2, y - PITCH_W / 2) <= CENTRE_RADIUS:
            return "kickoff"
        if fc_role == "goalkeeper" or (xa <= GK_RESTART_MAX_X_ATT and abs(y - PITCH_W / 2) <= BOX_HALF_WIDTH):
            return "gk_restart"
        if y <= TOUCHLINE_MARGIN or y >= PITCH_W - TOUCHLINE_MARGIN:
            return "throw_in"
        return "other_restart"
    prev = poss[ctx][poss[ctx] >= 0]
    if len(prev) and prev[-1] == OTHER[team]:
        return "regain"
    if len(prev) and prev[-1] == team and zone[s - 1] == 0:
        return "recycle"
    if (~visual_ok[ctx]).any():
        return "after_cut"
    return "unknown"


def classify_end(team, e):
    if e >= N - 1:
        return "match_end"
    for f in range(e + 1, min(N, e + 1 + nfr(CONTEXT_SEC))):
        if stop[f]:
            return "stoppage"
        if not visual_ok[f]:
            return "camera_cut"
        if poss[f] == OTHER[team]:
            return "possession_lost"
        if poss[f] == team and zone[f] == 0:
            return "reached_attacking_third"
    return "possession_unknown"


VIS_COL = {t: f"n_visible_team{t}" for t in TEAMS}
VIS = {t: B[VIS_COL[t]].to_numpy(float) for t in TEAMS}
T_SEC = B.t_sec.to_numpy(float)


def describe_phase(team, s, e, start_type=None, end_reason=None, n_merged=0):
    team, s, e = int(team), int(s), int(e)
    sl = slice(s, e + 1)
    xa, ya = to_att(bx[sl], by[sl], team)
    known = np.flatnonzero(~np.isnan(xa))
    i0, i1 = (known[0], known[-1]) if len(known) else (None, None)
    sx, sy = (xa[i0], ya[i0]) if i0 is not None else (np.nan, np.nan)
    ex, ey = (xa[i1], ya[i1]) if i1 is not None else (np.nan, np.nan)

    own = (car_id[sl] >= 0) & (car_team[sl] == team)
    ids = car_id[sl][own]
    seq = ids[np.r_[True, ids[1:] != ids[:-1]]] if len(ids) else np.array([], dtype=int)
    fc = int(seq[0]) if len(seq) else None
    fci = ID_INFO.get(fc, {}) if fc is not None else {}

    d = {
        "team": team, "team_name": TEAM_NAME[team],
        "def_team": OTHER[team], "def_team_name": TEAM_NAME[OTHER[team]],
        "start_frame": s, "end_frame": e,
        "t_start": T_SEC[s], "t_end": T_SEC[e], "clock_start": mmss(T_SEC[s]),
        "duration_s": round((e - s + 1) / FPS, 2),
        "start_type": start_type or classify_start(team, s),
        "end_reason": end_reason or classify_end(team, e),
        "start_x_att": sx, "start_y_att": sy, "start_third": third_of(sx), "start_lane": lane_of(sy),
        "end_x_att": ex, "end_y_att": ey, "end_third": third_of(ex), "end_lane": lane_of(ey),
        "min_x_att": np.nanmin(xa) if len(known) else np.nan,
        "max_x_att": np.nanmax(xa) if len(known) else np.nan,
        "first_carrier_id": fc, "first_carrier_name": fci.get("name"),
        "first_carrier_number": fci.get("number"), "first_carrier_role": fci.get("role"),
        "n_team_carriers": int(len(set(seq.tolist()))),
        "n_carrier_changes": int(max(0, len(seq) - 1)),
        "carrier_sequence": " > ".join(str(ID_INFO.get(int(c), {}).get("name", c)) for c in seq[:15])
                            + (" > ..." if len(seq) > 15 else ""),
        "nocarrier_before_s": round(nocar_len[s - 1] / FPS, 2) if s > 0 else 0.0,
        "valid_frac": valid[sl].mean(),
        "poss_known_raw_frac": (poss_raw[sl] >= 0).mean(),
        "ball_real_frac": (pos_src[sl] == "ball").mean(),
        "ball_pos_frac": len(known) / (e - s + 1),
        "carrier_frac": (car_id[sl] >= 0).mean(),
        "vis_att_mean": VIS[team][sl].mean(),
        "vis_def_mean": VIS[OTHER[team]][sl].mean(),
        "n_merged": n_merged,
    }
    d["quality_ok"] = bool(d["ball_pos_frac"] >= MIN_BALL_POS_FRAC and d["vis_def_mean"] >= MIN_DEF_VISIBLE_MEAN)
    return d

## Auto phase table + summary

In [ ]:
auto = pd.DataFrame([describe_phase(s["team"], s["start"], s["end"], n_merged=s["n_merged"]) for s in segments])
auto.insert(0, "phase_id", np.arange(1, len(auto) + 1))
auto.insert(1, "phase_uid", [f"{r.team_name}_{r.start_frame}_{r.end_frame}" for r in auto.itertuples()])
auto.to_csv(AUTO_PATH, index=False)
print(f"{len(auto)} phases -> {AUTO_PATH}")

summary = auto.groupby("team_name").agg(
    phases=("phase_id", "size"),
    total_min=("duration_s", lambda s: round(s.sum() / 60, 1)),
    median_s=("duration_s", "median"),
    max_s=("duration_s", "max"),
    quality_ok_pct=("quality_ok", lambda s: round(100 * s.mean(), 1)),
)
poss_valid_min = {TEAM_NAME[t]: round(((poss == t) & valid).sum() / FPS / 60, 1) for t in TEAMS}
summary["possession_min_usable"] = summary.index.map(poss_valid_min)
display(summary)
display(pd.crosstab(auto.team_name, auto.start_type, margins=True))
display(pd.crosstab(auto.team_name, auto.end_reason, margins=True))
display(pd.crosstab([auto.team_name, auto.start_third], auto.start_lane, margins=True))
display(auto.duration_s.describe().round(2).to_frame().T)

## Timeline check
Light bars = cleaned possession, dark bars = build-up phases, grey = stoppages.

In [ ]:
def bars(mask):
    rt = run_table(mask.astype(int))
    rt = rt[rt.value == 1]
    return [(T_SEC[s] / 60, (e - s + 1) / FPS / 60) for s, e in zip(rt.start, rt.end)]

COLORS = {TEAMS[0]: "#c0392b", TEAMS[1]: "#1f4e9c"}
n_panels = 3
span = T_SEC[-1] / 60 / n_panels
fig, axes = plt.subplots(n_panels, 1, figsize=(18, 2.2 * n_panels))
phase_mask = {t: np.zeros(N, bool) for t in TEAMS}
for r in auto.itertuples():
    phase_mask[r.team][r.start_frame:r.end_frame + 1] = True
for k, ax in enumerate(axes):
    for row, t in enumerate(TEAMS):
        y0 = row * 1.2
        ax.broken_barh(bars(poss == t), (y0, 1.0), color=COLORS[t], alpha=0.2)
        ax.broken_barh(bars(phase_mask[t]), (y0 + 0.2, 0.6), color=COLORS[t])
    ax.broken_barh(bars(stop), (-0.4, 0.3), color="grey")
    ax.set_xlim(k * span, (k + 1) * span)
    ax.set_yticks([0.5, 1.7])
    ax.set_yticklabels([TEAM_NAME[t] for t in TEAMS])
    ax.set_xlabel("match minute")
plt.tight_layout()
plt.show()

## Review sync
Writes `phases_review.csv`. Fill these columns (leave empty = keep as detected):
- `decision`: `keep`, `reject` or `edit`
- `edit_team` (team name), `edit_start_frame`, `edit_end_frame`, `edit_start_type`, `edit_end_reason`, `note`

Missed phases go into `phases_added.csv` (`team_name, start_frame, end_frame, start_type, end_reason, note`).
Existing decisions are carried over on re-runs; the previous file is backed up first.

In [ ]:
SHOW_COLS = ["phase_id", "phase_uid", "team_name", "clock_start", "duration_s", "start_frame", "end_frame",
             "start_type", "end_reason", "start_third", "start_lane", "end_third", "first_carrier_name",
             "quality_ok"]


def read_csv_any(path):
    return pd.read_csv(path, dtype=str, sep=None, engine="python").fillna("")


def sync_review(auto_df):
    new = auto_df[SHOW_COLS].copy()
    for c in REVIEW_COLS:
        new[c] = ""
    if REVIEW_PATH.exists():
        old = read_csv_any(REVIEW_PATH)
        for c in REVIEW_COLS:
            if c not in old:
                old[c] = ""
        reviewed = old[(old[REVIEW_COLS].apply(lambda s: s.str.strip()) != "").any(axis=1)]
        used = set()
        for i, r in new.iterrows():
            best, best_iou = None, 0.0
            for j, o in reviewed[reviewed.team_name == r.team_name].iterrows():
                if j in used:
                    continue
                iou = interval_iou(r.start_frame, r.end_frame, int(o.start_frame), int(o.end_frame))
                if iou > best_iou:
                    best, best_iou = j, iou
            if best is not None and best_iou >= REVIEW_MATCH_IOU:
                new.loc[i, REVIEW_COLS] = reviewed.loc[best, REVIEW_COLS].to_numpy()
                used.add(best)
        backup = OUT_DIR / f"phases_review_backup_{time.strftime('%Y%m%d_%H%M%S')}.csv"
        shutil.copy(REVIEW_PATH, backup)
        orphans = reviewed.drop(index=list(used))
        print(f"carried over {len(used)} reviewed rows; backup: {backup.name}")
        if len(orphans):
            op = OUT_DIR / "phases_review_orphans.csv"
            orphans.to_csv(op, index=False)
            print(f"WARNING: {len(orphans)} reviewed rows had no matching phase -> {op.name}")
    new.to_csv(REVIEW_PATH, index=False)
    if not ADDED_PATH.exists():
        pd.DataFrame(columns=["team_name", "start_frame", "end_frame", "start_type", "end_reason", "note"]) \
          .to_csv(ADDED_PATH, index=False)
    print(f"review file: {REVIEW_PATH}")
    return new


review = sync_review(auto)
review.head()

## Apply review → `phases_final.parquet`
Run after editing the review files. Edited and added phases get all descriptors recomputed.

In [ ]:
def _int_or_none(v):
    v = str(v).strip()
    return int(float(v)) if v not in ("", "nan", "None") else None


def _str_or_none(v):
    v = str(v).strip()
    return v if v not in ("", "nan", "None") else None


def _team(v):
    v = _str_or_none(v)
    if v is None:
        return None
    if v in NAME_TO_TEAM:
        return NAME_TO_TEAM[v]
    return int(float(v))


def apply_review():
    rev = read_csv_any(REVIEW_PATH)
    rows, problems = [], []
    for r in rev.itertuples(index=False):
        dec = str(r.decision).strip().lower()
        if dec in ("reject", "r", "x", "no"):
            continue
        team = _team(r.edit_team) if _team(r.edit_team) is not None else _team(r.team_name)
        s = _int_or_none(r.edit_start_frame)
        s = int(r.start_frame) if s is None else s
        e = _int_or_none(r.edit_end_frame)
        e = int(r.end_frame) if e is None else e
        if not (0 <= s <= e < N):
            problems.append(f"{r.phase_uid}: bad frames {s}-{e}")
            continue
        edited = any(_str_or_none(getattr(r, c)) for c in REVIEW_COLS[1:6])
        d = describe_phase(team, s, e, _str_or_none(r.edit_start_type), _str_or_none(r.edit_end_reason))
        d.update(origin="edited" if edited else "auto", reviewed=dec != "",
                 source_uid=r.phase_uid, note=_str_or_none(r.note))
        rows.append(d)
    if ADDED_PATH.exists():
        for r in read_csv_any(ADDED_PATH).itertuples(index=False):
            s, e, team = _int_or_none(r.start_frame), _int_or_none(r.end_frame), _team(r.team_name)
            if s is None or e is None or team is None or not (0 <= s <= e < N):
                problems.append(f"added row bad: {r}")
                continue
            d = describe_phase(team, s, e, _str_or_none(r.start_type), _str_or_none(r.end_reason))
            d.update(origin="added", reviewed=True, source_uid=None, note=_str_or_none(r.note))
            rows.append(d)
    final = pd.DataFrame(rows).sort_values("start_frame").reset_index(drop=True)
    final.insert(0, "phase_id", np.arange(1, len(final) + 1))

    # overlap checks
    for t in TEAMS:
        ft = final[final.team == t]
        ov = (ft.start_frame.to_numpy()[1:] <= ft.end_frame.to_numpy()[:-1]).sum()
        if ov:
            problems.append(f"{TEAM_NAME[t]}: {ov} overlapping phases")
    s_, e_ = final.start_frame.to_numpy(), final.end_frame.to_numpy()
    cross = int((s_[1:] <= e_[:-1]).sum())
    if cross:
        problems.append(f"{cross} overlaps between consecutive phases (any team)")

    final.to_parquet(FINAL_PATH, index=False)
    final.to_csv(FINAL_PATH.with_suffix(".csv"), index=False)
    print(f"{len(final)} phases -> {FINAL_PATH}")
    print(final.origin.value_counts().to_string())
    for p in problems:
        print("WARNING:", p)
    return final


final = apply_review()
display(pd.crosstab(final.team_name, final.start_type, margins=True))

## Review clips (optional)
One mp4 per phase with 2 s padding: banner with phase info, green border while inside the phase,
yellow circle on real ball detections only, small flipped triangle above the carrier, progress bar.
If the circle is not on the ball, seeking is off: set `MAX_GRAB_FRAMES = None` to decode sequentially.

Lanes note: open one clip whose `start_lane` is not `centre` and check the side matches. If it is mirrored,
set `LEFT_IS_LOW_Y_ATT = False` and rerun.

In [ ]:
import cv2

P_frames = P.frame_idx.to_numpy()
P_ptr = np.searchsorted(P_frames, np.arange(N + 1))
P_tid = P.track_id.to_numpy()
P_box = P[["x1", "y1", "x2", "y2"]].to_numpy(float)
BALL_PX = B[["ball_px_x", "ball_px_y"]].to_numpy(float)
MAX_GRAB_FRAMES = 25 * 120   # skip forward by decoding if the next clip starts within this many frames


def _draw(frame, f, r, scale, s, e, f0, f1):
    h, w = frame.shape[:2]
    inside = s <= f <= e
    if inside:
        cv2.rectangle(frame, (0, 0), (w - 1, h - 1), (60, 200, 60), 4)
    cv2.rectangle(frame, (0, 0), (w, 26), (0, 0, 0), -1)
    txt = (f"#{r.phase_id} {r.team_name} build-up | {r.start_type} -> {r.end_reason} | "
           f"{r.start_third}/{r.start_lane} | {r.clock_start} | {r.duration_s:.1f}s")
    cv2.putText(frame, txt, (8, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.putText(frame, f"frame {f}  {mmss(T_SEC[f])}", (8, h - 14), cv2.FONT_HERSHEY_SIMPLEX, 0.45,
                (255, 255, 255), 1, cv2.LINE_AA)
    if ball_src[f] == "detected" and not np.isnan(BALL_PX[f, 0]):
        c = (int(BALL_PX[f, 0] * scale), int(BALL_PX[f, 1] * scale))
        cv2.circle(frame, c, max(4, int(9 * scale)), (0, 255, 255), 2, cv2.LINE_AA)
    if car_id[f] >= 0:
        a, b = P_ptr[f], P_ptr[f + 1]
        hit = np.flatnonzero(P_tid[a:b] == car_id[f])
        if len(hit):
            x1, y1, x2, _ = P_box[a + hit[0]] * scale
            cx, top = int((x1 + x2) / 2), int(y1) - 4
            sz = max(4, int(8 * scale))
            tri = np.array([[cx - sz, top - sz], [cx + sz, top - sz], [cx, top]], np.int32)
            cv2.fillPoly(frame, [tri], (0, 255, 255))
    # progress bar
    y = h - 6
    cv2.line(frame, (0, y), (w, y), (90, 90, 90), 4)
    span_ = max(1, f1 - f0)
    xs, xe = int((s - f0) / span_ * w), int((e - f0) / span_ * w)
    cv2.line(frame, (xs, y), (xe, y), (60, 200, 60), 4)
    cv2.circle(frame, (int((f - f0) / span_ * w), y), 4, (255, 255, 255), -1)


def export_phase_clips(phases, phase_ids=None, pad_sec=2.0, scale=0.5, overwrite=False):
    CLIP_DIR.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    assert cap.isOpened(), f"cannot open {VIDEO_PATH}"
    vw, vh = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    w, h = int(vw * scale) // 2 * 2, int(vh * scale) // 2 * 2
    rows = phases if phase_ids is None else phases[phases.phase_id.isin(phase_ids)]
    rows = rows.sort_values("start_frame")
    pos = 0   # index of the next frame the decoder will return
    pad = nfr(pad_sec)
    t0 = time.time()
    for k, r in enumerate(rows.itertuples(), 1):
        out = CLIP_DIR / f"phase_{r.phase_id:03d}_{r.team_name}_{r.start_type}.mp4"
        if out.exists() and not overwrite:
            continue
        s, e = int(r.start_frame), int(r.end_frame)
        f0, f1 = max(0, s - pad), min(N - 1, e + pad)
        if f0 >= pos and (MAX_GRAB_FRAMES is None or f0 - pos <= MAX_GRAB_FRAMES):
            for _ in range(f0 - pos):
                cap.grab()
        else:
            cap.set(cv2.CAP_PROP_POS_FRAMES, f0)
        pos = f0
        vwr = cv2.VideoWriter(str(out), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (w, h))
        for f in range(f0, f1 + 1):
            ok, frame = cap.read()
            if not ok:
                break
            pos = f + 1
            if scale != 1.0:
                frame = cv2.resize(frame, (w, h), interpolation=cv2.INTER_AREA)
            _draw(frame, f, r, scale, s, e, f0, f1)
            vwr.write(frame)
        vwr.release()
        print(f"[{k}/{len(rows)}] {out.name}  ({time.time() - t0:.0f}s)")
    cap.release()


# examples:
# export_phase_clips(auto, phase_ids=[1, 2, 3])        # check alignment + lanes first
# export_phase_clips(auto)                               # all phases (long on CPU)
# export_phase_clips(auto[~auto.quality_ok])            # only the doubtful ones